# Evaluación de clustering / tipología (v2)

Aislado de `notebooks/eval_embeddings_v2.ipynb`: toda la evaluación de las
configuraciones de clustering sobre el embedding `z` del
`land2vec.model.TrajectoryAutoencoder`, en las 7 zonas out-of-domain.

**Este notebook consume, no produce.** El barrido y la selección corren fuera,
con `scripts/tune_clustering.py`:

1. `python scripts/tune_clustering.py --sweep {kmeans,gmm,hdbscan,hierarchical}`
   (con `--k-values` hasta ~120 para que las familias paramétricas tengan un
   contrincante al `k` efectivo de HDBSCAN).
2. `python scripts/tune_clustering.py --select`.

Eso deja `models/cluster_v2/summary.csv`,
`models/cluster_v2/chosen{,_medium,_coarse}{,_parametric}.json` y los
`data/clusters_{dynamic,pooled_subsampled}*.zip` que leen las celdas de abajo.

**Prerrequisito de modelo**: `models/autoencoder_v2/` (el autoencoder final
elegido -- ver `notebooks/eval_embeddings_v2.ipynb` §1). Si no existe, correr
primero `scripts/train_autoencoder.py`.


In [1]:
!git clone https://github.com/gefero/land2vec.git
%cd land2vec
!pip install -e . --quiet

Cloning into 'land2vec'...
remote: Enumerating objects: 774, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 774 (delta 4), reused 8 (delta 4), pack-reused 755 (from 1)
Receiving objects: 100% (774/774), 155.36 MiB | 4.26 MiB/s, done.
Resolving deltas: 100% (366/366), done.
Updating files: 100% (157/157), done.
/media/grosati/Elements1/PEN/Datasets_ML/land2vec/land2vec


In [2]:
import dataclasses
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from land2vec.dataset import SequenceDatasetAutoencoder
from land2vec.tokenizer import Tokenizer
from land2vec.utils import load_config, load_model

DATA_DIR = Path("data")
MODELS_DIR = Path("models")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ZONES = [
    "puna_noa",
    "patagonia_estepa",
    "periurbano_cordoba",
    "ibera",
    "delta_parana",
    "pampa_nucleo",
    "misiones_selva",
]
VALID_LABELS = [i for i in Tokenizer.VOCAB.values() if i != Tokenizer.VOCAB["[UNK]"]]


In [3]:
def load_zone_seqs(zone: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / f"id_seqs_text_2000_2022_{zone}.zip")


def load_zone_coords(zone: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / f"lat_long_df_{zone}.zip", usecols=["ID", "latitude", "longitude"])


@torch.inference_mode()
def encode_zone(model, seqs: pd.Series, batch_size: int = 1024) -> np.ndarray:
    "z (N, embed_dim) para todas las secuencias de una zona, en su orden original."
    loader = DataLoader(SequenceDatasetAutoencoder(seqs), shuffle=False, batch_size=batch_size)
    model.eval()
    chunks = [model.encode(x.to(DEVICE)).cpu() for x, _ in loader]
    return torch.cat(chunks).numpy()


def had_transition(seq: str) -> bool:
    "True si la secuencia tiene al menos un cambio de estado en los 23 años."
    return len(set(seq.split("-"))) > 1


## 1. Preparación: modelo final y embedding `z`


In [4]:
final_config = dataclasses.replace(load_config(MODELS_DIR / "autoencoder_v2"), device=DEVICE)
final_model = load_model(final_config, MODELS_DIR / "autoencoder_v2").to(DEVICE)
print(f"embed_dim={final_config.embed_dim} n_layer={final_config.n_layer} pooling={final_config.pooling}")

zone_seqs = {zone: load_zone_seqs(zone)["seqs"] for zone in ZONES}
pooled_seqs = pd.concat(zone_seqs.values(), ignore_index=True)

pooled_z = np.concatenate([encode_zone(final_model, zone_seqs[zone]) for zone in ZONES])
zone_labels = np.concatenate([[zone] * len(zone_seqs[zone]) for zone in ZONES])
pooled_ids = np.concatenate([load_zone_seqs(zone)["ID"].values for zone in ZONES])

dynamic_mask = pooled_seqs.apply(had_transition).values
print(f"secuencias con transición: {dynamic_mask.sum():,} / {len(dynamic_mask):,} ({dynamic_mask.mean()*100:.1f}%)")


embed_dim=8 n_layer=2 pooling=query


100%|██████████| 388800/388800 [00:05<00:00, 68478.64it/s]


secuencias con transición: 107,362 / 3,344,976 (3.2%)


## 2. Criterio de selección

`k=8` con KMeans sobre `z` sin estandarizar fue, en la ronda anterior, una
elección arbitraria -- el silhouette se calculó *después*, como reporte, no
como criterio de selección. Este notebook consume el resultado de
`scripts/tune_clustering.py` (`src/land2vec/cluster.py`): un barrido sobre 4
familias (KMeans, GaussianMixture, HDBSCAN, jerárquico/aglomerativo) x
preprocesado de `z` (crudo / estandarizado / L2), elegidas por un criterio
explícito -- ver `docs/v2_autoencoder_training.md` §7.2 para el detalle
completo:

1. Métricas internas (silhouette, Calinski-Harabasz, Davies-Bouldin).
2. **Estabilidad**: ARI entre reajustes sobre pares de submuestras al 80%
   (bootstrap) -- la defensa contra un `k` que solo se sostiene por el azar
   de la muestra, que la ronda anterior no tenía.
3. **Fidelidad del prototipo**: macro F1 (restringido a clases con
   soporte>0) de comparar cada secuencia real contra la trayectoria
   prototípica decodificada de su cluster -- un cluster solo sirve si su
   prototipo es una descripción honesta de sus miembros, no solo si está
   bien separado.
4. Coherencia espacial vs. una línea de base de etiquetas permutadas.

El jerárquico se ajusta sobre una submuestra estratificada por zona y se
extiende al resto por centroide más cercano (Ward directo sobre las 107k
filas resultó infeasible en el entorno de cómputo -- ver el docstring de
`land2vec.cluster.run_hierarchical_config`).

Corre sobre las secuencias con al menos una transición (3,2% del pool, filtro
ya justificado en la ronda anterior) y, para que el mapa no quede 96,8%
vacío, también etiqueta el pool completo con las constantes submuestreadas
al 15% (`land2vec.cluster.load_pool_subsampled`).

`scripts/tune_clustering.py --select` elige, con el mismo criterio, **6 configs**: la matriz de 3 granularidades (**fina** sin tope de `k` / **media** `k_effective <= 40` / **gruesa** `k_effective <= 20`) x 2 familias (**HDBSCAN** / **no-HDBSCAN**, lo mejor de KMeans/GMM/jerárquico) -- ver la comparación más abajo.


In [5]:
import json
from IPython.display import display

CLUSTER_DIR = MODELS_DIR / "cluster_v2"
cluster_summary = pd.read_csv(CLUSTER_DIR / "summary.csv")

# --select elige una config por celda de la matriz 3 granularidades x 2 familias
# (ver run_select en scripts/tune_clustering.py).
LEVEL_SPECS = [
    ("fina / HDBSCAN",      ""),
    ("fina / no-HDBSCAN",   "_parametric"),
    ("media / HDBSCAN",     "_medium"),
    ("media / no-HDBSCAN",  "_medium_parametric"),
    ("gruesa / HDBSCAN",    "_coarse"),
    ("gruesa / no-HDBSCAN", "_coarse_parametric"),
]
chosen_cfgs = {name: json.loads((CLUSTER_DIR / f"chosen{suffix}.json").read_text())
               for name, suffix in LEVEL_SPECS}

for name, c in chosen_cfgs.items():
    m = c["metrics"]
    print(f"{name:22s} {c['algo']:8s} space={str(c['space']):9s} k={m['k_effective']:>3d}  "
          f"sil={m['silhouette_mean']:.3f} stab={m['stability_ari']:.3f} "
          f"proto={m['prototype_fidelity']:.3f} spat={m['spatial_coherence']:.3f} "
          f"ruido={m['noise_frac']:.3f}  fallback={c['fallback']}")

STABILITY_THRESHOLD = chosen_cfgs["fina / HDBSCAN"]["stability_threshold"]
elig = cluster_summary[cluster_summary["eligible"]].dropna(subset=["stability_ari", "prototype_fidelity"])
stable = elig[elig["stability_ari"] >= STABILITY_THRESHOLD]
ranking_cols = ["run_id", "k_effective", "silhouette_mean", "stability_ari",
                "prototype_fidelity", "spatial_coherence", "noise_frac"]
print(f"\n{len(stable)}/{len(elig)} configs elegibles con stability_ari >= {STABILITY_THRESHOLD} "
      f"-- top 10 por prototype_fidelity, HDBSCAN vs. resto:")
display(stable[stable["algo"] == "hdbscan"][ranking_cols]
        .sort_values("prototype_fidelity", ascending=False).head(10))
stable[stable["algo"] != "hdbscan"][ranking_cols].sort_values("prototype_fidelity", ascending=False).head(10)


FileNotFoundError: [Errno 2] No such file or directory: 'models/cluster_v2/chosen_parametric.json'

### Seis configuraciones: 3 granularidades x 2 familias

`--select` elige una config por cada celda de {**fina** (sin tope de `k`) / **media**
(`k <= 40`) / **gruesa** (`k <= 20`)} x {**HDBSCAN** / **no-HDBSCAN**}, con el mismo
criterio (compuerta de `stability_ari >= 0.75`, luego máxima `prototype_fidelity`;
desempate por silhouette y menor `k`). La segunda columna existe porque, al extender el
barrido de `k` más allá de 20, las familias paramétricas dejaron de ser un
también-corrió: **GMM `diag` sobre `z` L2 llega a `prototype_fidelity` ~0.90 a `k=120`
con 0% de ruido**, contra 0.955 de HDBSCAN (que descarta ~10% como ruido y mide su
fidelidad solo sobre el resto).

| nivel | algo | k | silhouette | stability_ari | prototype_fidelity | ruido |
|---|---|---:|---:|---:|---:|---:|
| fina / HDBSCAN | HDBSCAN L2 | 118 | 0.91 | 0.91 | **0.96** | 9.8% |
| fina / no-HDBSCAN | GMM diag L2 | 120 | 0.74 | 0.92 | **0.90** | 0% |
| media / HDBSCAN | HDBSCAN std | 31 | 0.80 | 0.91 | 0.87 | 24% |
| media / no-HDBSCAN | GMM diag L2 | 40 | 0.55 | 0.82 | 0.72 | 0% |
| gruesa / HDBSCAN | HDBSCAN L2 | 17 | 0.58 | **0.72** | 0.74 | 20% |
| gruesa / no-HDBSCAN | GMM full L2 | 18 | 0.47 | 0.78 | 0.58 | 0% |

- **`prototype_fidelity` de GMM `diag` sube monótono con `k`** (0.61 a `k=20` -> 0.90 a
  `k=120`) y su `stability_ari` *también* (0.67 -> 0.92). KMeans y jerárquico se estancan
  en ~0.6 aun a `k=120`: es la varianza por componente de GMM `diag` la que le permite
  aislar en clusters puros los bloques de trayectorias idénticas (muy frecuentes en esta
  base). Queda abierto si eso es una tipología genuina o una tabla de patrones frecuentes
  -- los prototipos decodificados de abajo ayudan a discriminar.
- **Nivel medio (`k~31`)**: el sweet spot de HDBSCAN -- `prototype_fidelity` 0.87 con
  `stability_ari` 0.91, en 31 tipos manejables. GMM al mismo techo se queda en 0.72.
- **Nivel grueso (`k~17`)**: HDBSCAN k=17 medía `stability_ari` 0.77 en el barrido
  (`n_boot=3`); **al reajustar con `n_boot=10` bajó a 0.72, debajo del umbral de 0.75**
  (`chosen_coarse.json` queda con `fallback=false` porque esa bandera refleja el criterio
  con las métricas del barrido, no el reajuste). Es el sobreajuste al ruido de una sola
  corrida que la estabilidad por bootstrap está pensada para exponer. Se documenta igual
  como tipología *exploratoria*.

**El `-1` en los archivos de etiquetas tiene dos significados** (ver docs §7.2, Nota
metodológica):

- `clusters_dynamic*.zip`: `-1` = ruido honesto de HDBSCAN (0 para las familias
  no-HDBSCAN, que asignan todos los puntos).
- `clusters_pooled_subsampled*.zip`: `-1` = "sin tipificar" -- puntos del pool más lejos
  de todo centroide que el percentil 95 de la distancia al centroide de los puntos
  no-ruido (calibrado por config, umbral guardado en `chosen*.json`). Aplica a las 6 y
  hace comparables los dos archivos.

In [6]:
def load_level(name: str, suffix: str, chosen_cfg: dict) -> dict:
    "Carga todo lo necesario para describir una celda de la matriz (`suffix` combina granularidad y familia, ver LEVEL_SPECS)."
    dyn_labels_df = pd.read_csv(DATA_DIR / f"clusters_dynamic{suffix}.zip")
    assert len(dyn_labels_df) == dynamic_mask.sum(), (
        f"clusters_dynamic{suffix}.zip no coincide con las secuencias con transición actuales -- "
        "¿hace falta re-correr scripts/tune_clustering.py --select?"
    )
    k = int(dyn_labels_df["cluster"].max()) + 1  # ids 0..k-1; -1 (ruido HDBSCAN) no cuenta

    pooled_index_df = pd.DataFrame({"ID": pooled_ids, "zone": zone_labels, "seqs": pooled_seqs.values})
    pooled_index_df["_row"] = np.arange(len(pooled_index_df))
    dyn_with_z = dyn_labels_df.merge(pooled_index_df, on=["ID", "zone"], how="left")
    assert dyn_with_z["_row"].notna().all()
    z_dyn = pooled_z[dyn_with_z["_row"].values.astype(int)]

    pooled_labels_df = pd.read_csv(DATA_DIR / f"clusters_pooled_subsampled{suffix}.zip")

    return {
        "name": name, "chosen": chosen_cfg, "k": k,
        "noise_frac": float((dyn_labels_df["cluster"] == -1).mean()),             # ruido HDBSCAN en dinámicas
        "pooled_untyped_frac": float((pooled_labels_df["cluster"] == -1).mean()),  # "sin tipificar" en el pool (umbral de distancia)
        "dyn_labels_df": dyn_labels_df, "dyn_with_z": dyn_with_z, "z_dyn": z_dyn,
        "pooled_labels_df": pooled_labels_df,
    }


levels = {name: load_level(name, suffix, chosen_cfgs[name]) for name, suffix in LEVEL_SPECS}

summary_tbl = pd.DataFrame([{
    "nivel": lvl["name"], "algo": lvl["chosen"]["algo"], "space": lvl["chosen"]["space"], "k": lvl["k"],
    **{m: lvl["chosen"]["metrics"][m] for m in
       ["silhouette_mean", "stability_ari", "prototype_fidelity", "spatial_coherence"]},
    "ruido_dinamicas": lvl["noise_frac"], "sin_tipificar_pool": lvl["pooled_untyped_frac"],
} for lvl in levels.values()]).round(3)
print(summary_tbl.to_string(index=False))

# composición cluster x zona solo para los niveles interpretables (k chico)
for name, lvl in levels.items():
    if lvl["k"] > 45:
        continue
    print(f"\n--- {name} (k={lvl['k']}) : composición por zona ---")
    print(pd.crosstab(lvl["dyn_labels_df"]["cluster"], lvl["dyn_labels_df"]["zone"], normalize="index").round(2))


NameError: name 'chosen_cfgs' is not defined

## 3. Mapas por nivel


In [ ]:
coords = pd.concat([load_zone_coords(zone).assign(zone=zone) for zone in ZONES], ignore_index=True)


def plot_level_map(name: str, lvl: dict) -> None:
    "Dos filas (dinámicas / pool con constantes submuestreadas), un panel por zona. En gris: ruido de HDBSCAN (-1) en las dinámicas y 'sin tipificar' (-1 por umbral de distancia) en el pool."
    coords_dyn = coords.merge(lvl["dyn_labels_df"], on=["ID", "zone"], how="inner")
    assert len(coords_dyn) == len(lvl["dyn_labels_df"])
    coords_pooled = coords.merge(lvl["pooled_labels_df"], on=["ID", "zone"], how="inner")
    k = lvl["k"]
    cmap = "tab20" if k > 10 else "tab10"

    fig, axes = plt.subplots(2, 7, figsize=(28, 8))
    for col, zone in enumerate(ZONES):
        for row, df in enumerate([coords_dyn, coords_pooled]):
            sub = df[df["zone"] == zone]
            gris = sub[sub["cluster"] == -1]
            clustered = sub[sub["cluster"] != -1]
            point_size = 2 if row == 0 else 1
            axes[row, col].scatter(gris["longitude"], gris["latitude"], c="lightgray", s=point_size, alpha=0.5)
            axes[row, col].scatter(clustered["longitude"], clustered["latitude"], c=clustered["cluster"], cmap=cmap, s=point_size, vmin=0, vmax=k - 1)
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        axes[0, col].set_title(zone, fontsize=9)

    axes[0, 0].set_ylabel("con transición", fontsize=10)
    axes[1, 0].set_ylabel("pool submuestreado\n(constantes incl. al 15%)", fontsize=10)
    fig.suptitle(f"Nivel {name} -- {lvl['chosen']['algo']}, k={k}  |  gris: "
                 f"{lvl['noise_frac']*100:.0f}% ruido (dinámicas) / "
                 f"{lvl['pooled_untyped_frac']*100:.0f}% sin tipificar (pool)")
    fig.tight_layout()
    slug = name.replace(" / ", "_").replace("-", "").lower()
    fig.savefig(Path("imgs") / f"v2_eval_cluster_map_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()


for name, lvl in levels.items():
    plot_level_map(name, lvl)


## 4. Prototipos decodificados


In [ ]:
from sklearn.metrics import f1_score


def prototype_table(lvl: dict) -> pd.DataFrame:
    """Por cluster: tamaño, fidelidad del prototipo (macro F1 restringido a clases con soporte>0),
    la trayectoria prototípica decodificada, y las 3 secuencias reales más cercanas al centroide."""
    raw_centers = np.array(lvl["chosen"]["raw_centers"])
    with torch.inference_mode():
        proto_logits = final_model.decode(torch.tensor(raw_centers, dtype=torch.float32, device=DEVICE))
        proto_tokens = proto_logits.argmax(-1).cpu()

    dyn_with_z = lvl["dyn_with_z"]
    z_dyn = lvl["z_dyn"]
    encoded_dyn = np.stack([Tokenizer.encode(s) for s in dyn_with_z["seqs"]])
    cluster_labels = dyn_with_z["cluster"].values

    rows = []
    for cluster_id in range(len(raw_centers)):
        mask = cluster_labels == cluster_id
        n_members = int(mask.sum())
        if n_members == 0:
            continue
        y_true = encoded_dyn[mask].reshape(-1)
        y_pred = np.tile(proto_tokens[cluster_id].numpy(), n_members)
        present = sorted(set(y_true.tolist()) & set(VALID_LABELS))
        fidelity = f1_score(y_true, y_pred, average="macro", labels=present, zero_division=0) if present else float("nan")

        dists = np.linalg.norm(z_dyn[mask] - raw_centers[cluster_id], axis=1)
        nearest_idx = np.argsort(dists)[:3]
        nearest_seqs = dyn_with_z.loc[mask, "seqs"].values[nearest_idx]

        rows.append({
            "cluster": cluster_id, "n": n_members, "prototype_fidelity": round(float(fidelity), 4),
            "prototipo": Tokenizer.decode(proto_tokens[cluster_id]),
            "vecino_real_1": nearest_seqs[0] if len(nearest_seqs) > 0 else None,
            "vecino_real_2": nearest_seqs[1] if len(nearest_seqs) > 1 else None,
            "vecino_real_3": nearest_seqs[2] if len(nearest_seqs) > 2 else None,
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False)


proto_tables = {name: prototype_table(lvl) for name, lvl in levels.items()}
for name, t in proto_tables.items():
    v = t.dropna(subset=["prototype_fidelity"])
    wmean = np.average(v["prototype_fidelity"], weights=v["n"]) if len(v) else float("nan")
    print(f"{name:22s} k={len(t):>3d}  fidelidad media ponderada por tamaño = {wmean:.3f}")

print(f"\nPrototipos -- fina / HDBSCAN (k={levels['fina / HDBSCAN']['k']}), 20 más numerosos:")
proto_tables["fina / HDBSCAN"].head(20)


In [ ]:
from IPython.display import display

for name in ["fina / no-HDBSCAN", "media / HDBSCAN", "media / no-HDBSCAN",
             "gruesa / HDBSCAN", "gruesa / no-HDBSCAN"]:
    t = proto_tables[name]
    print(f"\n=== Prototipos -- {name} (k={len(t)}) ===")
    display(t.head(15))


## Conclusiones

La ronda anterior fijaba `k=8` con KMeans sobre `z` sin estandarizar de forma
arbitraria (silhouette 0.43 se calculaba *después*, como reporte). Esta ronda lo
reemplaza por un criterio explícito (compuerta de `stability_ari >= 0.75`, luego
máxima `prototype_fidelity`; desempate por silhouette y menor `k`), aplicado a
cada celda de la matriz 3 granularidades x 2 familias.

**`prototype_fidelity` de GMM `diag` sube monótono con `k`** (0.61 a `k=20` ->
0.90 a `k=120`) y su `stability_ari` *también* (0.67 -> 0.92). KMeans y
jerárquico se estancan en ~0.6 aun a `k=120`: es la varianza por componente de
GMM `diag` la que le permite aislar en clusters puros los bloques de
trayectorias idénticas (muy frecuentes en esta base). Queda abierto si eso es
una tipología genuina o una tabla de patrones frecuentes -- los prototipos
decodificados de arriba ayudan a discriminar.

**Nivel medio (`k~31`)**: el sweet spot de HDBSCAN -- `prototype_fidelity` 0.87
con `stability_ari` 0.91, en 31 tipos manejables.

**Nivel grueso (`k~17`)**: HDBSCAN medía `stability_ari` 0.77 en el barrido
(`n_boot=3`); al reajustar con `n_boot=10` bajó a 0.72, debajo del umbral. Es el
sobreajuste al ruido de una sola corrida que la estabilidad por bootstrap está
pensada para exponer -- se documenta igual como tipología *exploratoria*.

Ver `docs/v2_autoencoder_training.md` §7.2 para el criterio completo y la
discusión metodológica del doble significado de `-1` (ruido de HDBSCAN en
`clusters_dynamic*.zip` vs. "sin tipificar" por umbral de distancia en
`clusters_pooled_subsampled*.zip`).
